## Confirmación del alcance: celdas vacías en tablas de Word

En 01_exploration.ipynb vimos una fila con MES='' en la primera tabla
del docx, que significa "sigue siendo el mes anterior" (patrón típico
de tablas pegadas desde Excel sin fusión real). Antes de escribir un
fix, comprobamos: ¿pasa en las DOS tablas del documento? ¿en más de
una columna? ¿cuántas filas afectadas en total?

In [ ]:
from pathlib import Path
import docx

DATA_DIR = Path.cwd().parent / "data" / "raw"
docx_path = DATA_DIR / "financiero_informe_1q.docx"

d = docx.Document(docx_path)

for i, tabla in enumerate(d.tables):
    print(f"--- Tabla {i} ({len(tabla.rows)} filas x {len(tabla.columns)} columnas) ---")
    for fila in tabla.rows:
        celdas = [c.text for c in fila.cells]
        if any(c.strip() == "" for c in celdas):
            print("  fila con celda(s) vacía(s):", celdas)

### Qué buscamos en esta salida

- ¿Está el patrón solo en la columna MES, o también en otras?
- ¿Aparece en la Tabla 1 (jornadas/programas) además de la Tabla 0?
- ¿Hay alguna fila con la PRIMERA celda vacía en la tabla entera
  (que no tendría "valor anterior" al que hacer forward-fill)?
  Ese caso habría que tratarlo aparte, no asumir que siempre hay
  un valor previo válido.

### Conclusión: el forward-fill debe aplicarse SOLO a la columna 0

No existe un patrón general de "celda vacía = repite la de arriba" en
estas tablas. Existe un patrón MUY concreto: la columna de agrupación
(columna 0, aquí "MES") se deja vacía cuando el valor es el mismo que
la fila anterior — un artefacto típico de exportar desde Excel a Word
sin fusión real de celdas.

Cualquier otra columna vacía (Nº JORNADAS, el total de la última
columna) es un dato ausente genuino y NO debe rellenarse. Forward-fill
scoped únicamente a índice de columna 0, nunca a la fila entera.

Esto es más estrecho que el "propagar celdas fusionadas" del Excel:
allí propagábamos todo el rango porque sabíamos con certeza (via
merged_cells.ranges) qué celdas compartían valor. Aquí no hay esa
certeza estructural — es una convención de la columna 0, así que
limitamos el arreglo a lo que podemos justificar con los datos que
hemos visto.

In [ ]:
def _forward_fill_columna_agrupadora(tabla_filas: list[list[str]], indice_columna: int = 0) -> list[list[str]]:
    """Rellena celdas vacías de UNA columna concreta con el último valor no
    vacío visto por encima, para reconstruir la columna de agrupación
    (ej. MES) cuando una tabla de Word la deja en blanco por continuidad
    en vez de repetir el valor. No toca ninguna otra columna."""
    ultimo_valor = None
    filas_corregidas = []
    for fila in tabla_filas:
        fila = list(fila)
        valor_actual = fila[indice_columna].strip() if fila[indice_columna] else ""
        if valor_actual:
            ultimo_valor = valor_actual
        elif ultimo_valor is not None:
            fila[indice_columna] = ultimo_valor
        filas_corregidas.append(fila)
    return filas_corregidas


tabla_0 = [[c.text for c in fila.cells] for fila in d.tables[0].rows]
tabla_0_corregida = _forward_fill_columna_agrupadora(tabla_0, indice_columna=0)

for original, corregida in zip(tabla_0, tabla_0_corregida):
    print(original, "->", corregida)

### Qué comprobar en la salida

1. La cabecera ('MES', 'JORNADA/PROGRAMA'...) no debe cambiar.
2. Las filas con MES='' deben quedar con el mes anterior correcto.
3. La fila TOTAL debe quedar EXACTAMENTE igual que antes (columna 0
   ya tenía valor, no se toca).
4. La fila con Nº JORNADAS='' debe seguir con esa celda vacía —
   si aparece rellenada, hay un bug en la función.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # asume que ejecutas desde notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src_agents.rag.extractor_generico import extraer_docx


In [ ]:
from pathlib import Path
DATA_DIR = Path.cwd().parent / "data" / "raw"

from src_agents.rag.extractor_generico import extraer_xlsx, extraer_docx

# Excel: celda de cabecera de grupo propagada
bloques_xlsx = extraer_xlsx(DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx")
print(bloques_xlsx[0].contenido[1])

# Word: columna MES rellenada
bloques_docx = extraer_docx(DATA_DIR / "financiero_informe_1q.docx")
tabla = [b for b in bloques_docx if b.tipo_bloque == "tabla"][0]
for fila in tabla.contenido:
    print(fila)